# 📌 Dataiku Project Setup & Utilities

This section initializes a **connection to a Dataiku DSS project** and imports all the necessary libraries for data processing, similarity matching, and PDF parsing.

---

## 🛠 Imports

- **dataikuapi** → Connects to Dataiku DSS API.  
- **typing (Dict, Any)** → Type hints for better code readability.  
- **os, sys, io, json, uuid, logging** → System utilities, file handling, JSON parsing, unique ID generation, and logging.  
- **cosine_similarity (scikit-learn)** → Measures similarity between numeric vectors (useful for ML/NLP tasks).  
- **rapidfuzz.fuzz** → High-performance fuzzy string matching.  
- **fitz (PyMuPDF) & pdfplumber** → Parse and extract structured data from PDF files.  
- **OpensearchUtil** → Custom utility to interact with OpenSearch (for vector/text search).  
- **get_dataiku_client_and_project** → Helper function (commented out) to get client and project.

---

## ⚙ Configuration

```python
DATAIKU_HOST = "http://10.45.152.66:10000"
API_SECRET_KEY = "dkuaps-***************"  # Keep secret
PROJECT_NAME = "ECSGENERATION"


In [20]:
import dataikuapi
from typing import Dict, Any
import os
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz
import io
import fitz  # PyMuPDF
import pdfplumber
from uuid import uuid4
from soa_extraction.opensearch_utils import OpensearchUtil

import sys
import os
import fitz  # PyMuPDF
# from langgraph_utils.Info_extractor import InfoExtractorAgent
# from langgraph_utils.chat_history import SnowflakeChatMessageHistory
import json
# from langgraph_utils.file_parser import FileParser
# from langgraph_utils.digitization import main_handler
import uuid
# from langgraph_utils import creds
# from langgraph_utils.variables import PROJECT_NAME, SECRET_NAME, TOKEN_KEY
from utils.connection import get_dataiku_client_and_project
import logging



DATAIKU_HOST = "http://10.45.152.66:10000"
API_SECRET_KEY = "dkuaps-b3EsRXVjU3w4y7nd4KEwibEr04CjFPZr"          
PROJECT_NAME = "ECSGENERATION"   

# client, proj = get_dataiku_client_and_project(PROJECT_NAME, SECRET_NAME, TOKEN_KEY)
client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
proj = client.get_project(PROJECT_NAME)

# 🧠 `make_llm_call` Function – LLM-Generated Edit Check Specifications

This function uses a **Large Language Model (LLM)** to generate **deterministic JSON edit-check specifications** for a given `form_name` and `field_name`.

---

## 📋 Function Overview

```python
def make_llm_call(form_name, field_name):
    ...


In [21]:
def make_llm_call(form_name,field_name):
    prompt = f"""
        You are a Edit check list specifcation specialist for Case Report Form .
        Given is an Example of how the data needs to be Generated this is a few shot example only
        `[
        vaildation_id : MVAL_DM027
        form_name : Demographics,
        'form_domain_name':DM
        'form_field_value':Country
        'variable_name': 'COUNTRY',
        'validation_logic': '(DM.COUNTRY is enterable and missing)', 'reasoning': 'Field must not be missing when enterable', 
        'action': 'prompt user with ACTION DETAILS',
        'action_details': '<query the field for missing data>',             
                ]`
        
        now you job is to create the json output for the these input fields :
        form name :{form_name} 
        field name :{field_name}
        
        Instruction:
        - Do not Expalin yourself , only json output
        - Do not change the data , generate for the form and field provided 
        - dont hallucinate 
        
    """
    prompt2 = f"""You are an Edit Check Specification Specialist for Case Report Forms (CRFs). You will receive a user input that contains two variables: form_name and field_name. Your ONLY job is to produce a JSON array of edit-check specification objects for the provided form_name and field_name. Follow these rules exactly:

1. OUTPUT FORMAT:
   - Return raw JSON only (no markdown, no code fences, no explanations, no extra text).
   - The top-level JSON must be an JSON. 
   - Generate Output for
   form name :{form_name} 
        field name :{field_name}

2. SCHEMA (each object must include exactly these keys):
   - validation_id
   - form_name
   - form_domain_name
   - form_field_value
   - variable_name
   - validation_logic
   - reasoning
   - action
   - action_details
   - source

   Do not add or remove keys.

4. DETERMINISTIC DERIVATIONS:
   - variable_name: derive by converting field_name to UPPERCASE snake_case (letters, numbers, underscores only). Example: "Date of Birth" -> "DATE_OF_BIRTH".
   - form_domain_name: map common forms (Demographics->DM, Medical History->MH, Vital Signs->VS, Adverse Event->AE, Concomitant Meds->CM, Informed Consent->IC, Physical Examination->PE, Laboratory->LB). If the form_name is not in the mapping, derive the domain by concatenating the first 2–3 letters of each significant word in the form_name and uppercasing (e.g., "Post-treatment Follow-up" -> "PTF").
   - validation_id: deterministic string "MVAL_{{form_name}}NNN" where NNN is a 3-digit sequence starting at 001. Use 001 unless other context is provided.

... (other rules unchanged) ...

7. NO HALLUCINATION:
   - Do not invent facts, values, mappings, or external knowledge not produced by the deterministic rules above.
   - If you cannot deterministically choose a validation_logic or domain from the input, do NOT invent — instead return this exact error structure (as the only element in the array):

     
       {{
         "error": "insufficient_input",
         "form_name": "{form_name}",
         "field_name": "{field_name}",
         "message": "Cannot generate validation logic deterministically for this field."
       }}
     

End of system instructions.
"""
    
    prompt3 = f"""
You are an Edit Check Specification Specialist for Case Report Forms (CRFs). You will receive a user input that contains two variables: `form_name` and `field_name`. Your ONLY job is to produce a JSON array of edit-check specification objects for the provided `form_name` and `field_name`.

Follow these rules exactly:

1. OUTPUT FORMAT:
   - Return raw JSON only (no markdown, no code fences, no explanations, no extra text).
   - The top-level JSON must be a JSON array.
   - Generate output for:
     - form name: {form_name}
     - field name:{field_name}

2. SCHEMA (Each Object Must Include Exactly These Keys):
   - validation_id
   - form_name
   - form_domain_name
   - form_field_value
   - variable_name
   - validation_logic
   - reasoning
   - action
   - action_details
   - source

   Do not add or remove keys.

3. DETERMINISTIC DERIVATIONS:
   - variable_name: Convert field_name to UPPERCASE snake_case (letters, numbers, underscores only).
     Example: "Date of Birth" -> "DATE_OF_BIRTH".
   - form_domain_name:
       - Use standard mappings:
           Demographics -> DM
           Medical History -> MH
           Vital Signs -> VS
           Adverse Event -> AE
           Concomitant Meds -> CM
           Informed Consent -> IC
           Physical Examination -> PE
           Laboratory -> LB
       - If no mapping exists, derive by concatenating the first 2–3 letters of each significant word in form_name and uppercasing.
         Example: "Post-treatment Follow-up" -> "PTF".
   - validation_id: Deterministic string "MVAL_{{form_name}}NNN" where NNN is a 3-digit sequence starting at 001. Use 001 unless other context is provided.

4. VALIDATION LOGIC RULES:
   Generate one or more validation objects based on the following deterministic rules (if applicable):

   1. Missing Field Check: If the field is user-enterable, ensure it is not blank.
   2. Future Date Check: If the field is a date, ensure it is not in the future.
   3. Non-Conformance Check: If the field has controlled values (dictionary, code list), ensure the value is valid.
   4. Visibility Check:
      - If top-level Y/N field "Has the subject had any adverse events since the last visit?" = Yes, dependent fields must be visible.
      - If No, dependent fields must be hidden.
   5. Conditional Completion:
      - If top-level Y/N = Yes, all dependent fields must be filled out.
      - If No and a "Reason" field exists, ensure "Reason" is not blank.
   6. Other Specify Check: If "Other" is selected, "Other Specify" must not be blank.
   7. Chronological Validations:
      - Visit date must not be before previous visit date.
      - Visit date must not be after Disposition date.
      - AE start date must not be before Informed Consent (IC) date.
      - MH start date must not be after IC date.
      - End date must not be before its corresponding start date.
   8. Cross-Form Checks:
      - If AE or MH is related to a CM record, ensure CM start date is not before event start date.

   For each validation object, clearly specify:
   - validation_logic: The exact check being applied (deterministic wording).
   - reasoning: Why the check is necessary (data quality, protocol compliance, etc.).
   - action: What should happen when the check fails (e.g., flag, warning, hard edit).
   - action_details: Specific user-facing message.
   - source: Always set to "Edit Check Specification Rule".

5. NO HALLUCINATION:
   Do not invent facts, values, or mappings.
   If a validation cannot be generated deterministically, provide NaN
   
"""


    default_llm_model = proj.get_variables()['local'].get('default_llm_model')
        #self.default_llm_model = 'azureopenai:Azure-OpenAi:gpt-4o'
#         logging.info(f"[PlannerAgent] Initialized with LLM model: {self.default_llm_model}")
       
    llm = proj.get_llm(default_llm_model).as_langchain_llm(
        completion_settings={
        "temperature": 0,
                
                "timeout": 300,
            "max_tokens": 8192
            # 5 minutes
            }
        )
        
    output = llm.invoke(prompt3)
    return output

 # 📑 CRF Digitization & Extraction of Form/Field Names

This section performs **digitization of a historical CRF (Case Report Form) PDF**, extracts form names and field names, and organizes them into a structured dataset for downstream processing.

---




In [418]:
import re
from typing import List, Dict, Any

class GenericFormFieldExtractor:
    """
    Generic extractor for any fields from clinical trial forms.
    No hardcoded field-specific logic - works with any field names.
    """
    
    def extract_fields(self, text: str, field_names: List[str]) -> Dict[str, Any]:
        """
        Extract any specified fields from form text.
        
        Args:
            text: The form text (single page or multiple pages)
            field_names: List of field names to extract (e.g., ['Date of Birth', 'Sex at Birth', 'Age'])
        
        Returns:
            Dictionary with field names as keys and their values/options
        """
        results = {}
        lines = text.strip().split('\n')
        
        for field_name in field_names:
            field_data = self._extract_field(field_name, lines)
            results[field_name] = field_data
        
        return results
    
    def _extract_field(self, field_name: str, lines: List[str]) -> Dict[str, Any]:
        """
        Extract a single field and its associated data.
        Returns field number and any values/options found.
        """
        field_lower = field_name.strip().lower()
        
        for i, line in enumerate(lines):
            # Check if this line contains the field name
            if field_lower in line.lower():
                field_info = {
                    'field_name': field_name,
                    'field_number': None,
                    'line_text': line.strip(),
                    'values': [],
                    'options': []
                }
                
                # Extract field number (usually at the end of the line)
                field_num_match = re.search(r'\s+(\d+)\s*$', line)
                if field_num_match:
                    field_info['field_number'] = field_num_match.group(1)
                
                # Look at subsequent lines for options/values
                field_info['options'] = self._extract_options(lines, i)
                
                # Extract selected/checked values (lines with numbers at the end)
                field_info['values'] = self._extract_selected_values(lines, i)
                
                return field_info
        
        return None
    
    def _extract_options(self, lines: List[str], start_idx: int, max_lines: int = 20) -> List[str]:
        """
        Extract all available options for a field.
        Options are typically indented text following the field name.
        """
        options = []
        
        for i in range(start_idx + 1, min(start_idx + max_lines, len(lines))):
            line = lines[i]
            
            # Stop if we hit another field (less indented text with content)
            if self._is_new_field(line):
                break
            
            # Check if line is an option (indented, has content)
            if self._is_option_line(line):
                option_text = self._clean_option_text(line)
                if option_text:
                    options.append(option_text)
        
        return options
    
    def _extract_selected_values(self, lines: List[str], start_idx: int, max_lines: int = 20) -> List[str]:
        """
        Extract selected/checked values (options with numbers indicating selection).
        """
        selected = []
        
        for i in range(start_idx, min(start_idx + max_lines, len(lines))):
            line = lines[i]
            
            # Stop if we hit another field
            if i > start_idx and self._is_new_field(line):
                break
            
            # Check if this line has a selection indicator (number at end or marked)
            if self._has_selection_indicator(line):
                value_text = self._extract_value_text(line)
                if value_text:
                    selected.append(value_text)
        
        return selected
    
    def _is_new_field(self, line: str) -> bool:
        """Check if line starts a new field."""
        # New fields typically start with less indentation and have substantial text
        stripped = line.strip()
        if not stripped:
            return False
        
        # Check indentation - new fields usually have 5-20 spaces
        leading_spaces = len(line) - len(line.lstrip())
        
        # New field if: moderate indentation, has letter start, and reasonable length
        if 5 <= leading_spaces <= 25 and stripped[0].isupper() and len(stripped) > 3:
            # Not a new field if heavily indented (likely an option)
            if leading_spaces > 30:
                return False
            return True
        
        return False
    
    def _is_option_line(self, line: str) -> bool:
        """Check if line is an option/choice."""
        stripped = line.strip()
        if not stripped:
            return False
        
        # Options are usually indented more than fields
        leading_spaces = len(line) - len(line.lstrip())
        
        return leading_spaces >= 20 and len(stripped) > 1
    
    def _clean_option_text(self, line: str) -> str:
        """Clean option text by removing numbers and extra whitespace."""
        text = line.strip()
        # Remove trailing numbers (field reference numbers)
        text = re.sub(r'\s+\d+\s*$', '', text)
        return text.strip()
    
    def _has_selection_indicator(self, line: str) -> bool:
        """Check if line has a selection indicator (letter + number or just number at end)."""
        # Pattern: text followed by single letter and/or number at end
        # Examples: "Male 4", "Y 1", "M 3"
        return bool(re.search(r'[A-Z]\s+\d+\s*$', line))
    
    def _extract_value_text(self, line: str) -> str:
        """Extract the value text from a selected option."""
        text = line.strip()
        # Remove the selection indicator (letter and/or number at end)
        text = re.sub(r'\s+[A-Z]?\s*\d+\s*$', '', text)
        return text.strip()
    
    def extract_all_fields(self, text: str) -> List[Dict[str, Any]]:
        """
        Auto-detect and extract ALL fields from the form.
        Useful when you don't know field names in advance.
        """
        all_fields = []
        lines = text.strip().split('\n')
        
        for i, line in enumerate(lines):
            if self._is_new_field(line):
                # Extract field name
                field_name = re.sub(r'\s+\d+\s*$', '', line.strip())
                if field_name and '?' not in field_name[-5:]:  # Skip question marks
                    field_name_clean = field_name.rstrip('?').strip()
                else:
                    field_name_clean = field_name.strip()
                
                if field_name_clean and len(field_name_clean) > 2:
                    field_data = self._extract_field(field_name_clean, lines)
                    if field_data:
                        all_fields.append(field_data)
        
        return all_fields




In [458]:
import os
import io
import logging
import re
from uuid import uuid4
from typing import List, Dict, Any

import traceback

import fitz  # PyMuPDF (not actually used here but could be if needed)
import pdfplumber


logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)


class HistoricalCRF:
    """
    Class to extract structured data from historical CRF PDFs stored in a Dataiku managed folder.
    """

    def __init__(self, client, proj, chunk_size: int = 1000):
        """
        Initialize the HistoricalCRF extractor.

        :param client: Dataiku API client
        :param proj: Dataiku project handle
        :param chunk_size: (Optional) Number of characters per chunk for downstream processing
        """
        self.proj = proj
        self.client = client
        self.chunk_size = chunk_size

        self.variables = proj.get_variables().get("local", {})
        self.s3_folder_dataset_id = self.variables.get("file_upload")
        if not self.s3_folder_dataset_id:
            raise ValueError("Missing 'file_upload' folder in project variables.")

        self.input_folder = proj.get_managed_folder(self.s3_folder_dataset_id)
        self.files = self.input_folder.list_contents().get("items", [])
        self.toc_page_limit = 20
        self.config = self.variables

        logger.info(f"📂 Found {len(self.files)} files in managed folder: {self.s3_folder_dataset_id}")

    def historical_mapping(self, file_path: str,file_id) -> List[Dict[str, Any]]:
        """
        Process a single PDF file, extracting headers and fields into structured data.

        :param file_path: Path of the file inside the managed folder.
        :return: List of dictionaries containing structured CRF data.
        """
        response = []
        oid_response = []
        back_track = []
        previous_pointer = 0
        prev_track = []
        try:
            with self.input_folder.get_file(file_path) as stream:
                file_bytes = stream.raw.data

            pdf_file_like = io.BytesIO(file_bytes)
            import pymupdf4llm
            import fitz
            pdf_document = fitz.open(stream = file_bytes, filetype = 'pdf')
            with pdfplumber.open(pdf_file_like) as pdf:
                total_pages = len(pdf.pages)
                snowflake_conn = self.config.get("snowflake_connection_string")
        
        
                #input_folder =  project.get_managed_folder(proj_vars.get("ecs").get("file_upload"))
                table_file_upload = self.config.get("ecs", {}).get("ecs_file_upload", {})
                field_names_list= []
                page_list = []
                page_idx = 0
                
                
                for page_num, page in enumerate(pdf.pages[1:], start=1):
                    try:
                        page_text = page.extract_text(layout=True)
                        
                        percent = (page_num / total_pages) * 100
                        
#                         update_query = f"""
#                                         UPDATE {table_file_upload}
#                                         SET "digitization_percent" = '{percent}'
#                                         WHERE "crf_file_id" = '{file_id}'
#                                          ;
#                                     """
#                         self.client.sql_query(query = update_query,connection = snowflake_conn,  post_queries=["COMMIT"])

                        if not page_text:
                            logger.debug(f"⚠️ Skipping empty page {page_num} in {file_path}")
                            continue

                        if "field name" in page_text.lower():
                            #table = page.extract_tables()
                            #print(page_text)
                            
                            result = self.extract_oids(page_text, file_path)
                            
                            if result:
                           
                                oid_response.append(result)
                            
                            continue
                            
                        
                            
                        result = self._process_page(page_text, file_path)
              
                        if result:
                           
                            response.append(result)
#                             page_list.append(page_idx)
#                             page_idx +=1
                           

                    except Exception as page_error:
                        logger.error(f"❌ Error processing page {page_num} in {file_path}: {page_error}")
                        t = traceback.format_exc()
                        return {"message":f"error caused ue to {t}"}

        except Exception as file_error:
            logger.error(f"❌ Error opening or processing file {file_path}: {file_error}")
            t = traceback.format_exc()
            return {"message":f"error caused ue to {t}"}

        
        return response,oid_response
    
    def extract_oids(self,page_text: str, file_path: str) -> Dict[str,Any]:
        
        
        lines = page_text.split("\n")
#         print(page_text)
        pattern = r'(?m)^\s*(\d+)\s+([A-Z0-9_]+)\s*\$?\d*'
        matches = re.findall(pattern, page_text)

        field_names_list = [{"field_number": int(num), "field_name": name} for num, name in matches]
        print(field_names_list)
#         pattern = r'(?m)^\s*\d+\s+([A-Z0-9_]+)\s*\$?\d*'
    
#         field_names_list = re.findall(pattern, page_text)
        
        #print(field_names_list)
        header_lines = []
        
        current_field = None
        in_field_section = False

        for line in lines:
            line = line.strip()
            if not line:
                continue

            # Detect start of fields section
            if not in_field_section:
                if "generated" in line.lower():
                    in_field_section = True
                    continue
                header_lines.append(line)
                continue

        header_text = " ".join([j for j in header_lines if "form" in j.lower() or "folder" in j.lower()]).strip()
        
        if field_names_list and header_text:
            return {
                
                "template_name": os.path.basename(file_path),
                "path": file_path,
                "id": str(uuid4()),
                "source_data": {
                    "assessments": header_text,
                    "fields_oid": field_names_list,
                },
            }

        return None
        
    def _process_page(self, page_text: str, file_path: str) -> Dict[str, Any]:
        """
        Process a single page's text, extracting header and field-value pairs.

        :param page_text: Extracted text from a PDF page
        :param file_path: The path of the file (for reference)
        :return: A dictionary with structured data or None if nothing found
        """
        
        #print(page_text)
        
        # Initialize extractor
        extractor = GenericFormFieldExtractor()

        # MAIN USE CASE: Auto-extract ALL fields from the document
        

        all_fields = extractor.extract_all_fields(page_text)

        
        final_fields = []
        for field in all_fields:
            if  field['field_number']:
                final_fields.append({
                    "field_name" : field['field_name'],
                    "field_number" : field['field_number']
                })
               

        
    
    
        lines = page_text.split("\n")
        header_lines = []
        field_value_map = {}
        current_field = None
        in_field_section = False
       
        
       
        field_pattern = re.compile(r'^(?P<field>.+?)(?:\s{2,})(?P<value>.+?)\s*$', re.MULTILINE)

#         matches = pattern.finditer(page_text)

#         for m in matches:
#             print(m.group("field").strip(), "=>", m.group("value").strip())

       
#         field_pattern = re.compile(r"""
#             ^(?P<field>.+?)            # Field name (non-greedy)
#             (?:\t|\s{2,})+            # Separator: tab or ≥2 spaces
#             (?P<value>.+?)            # Field value
#             \s*$                      # Optional trailing spaces
#         """, re.VERBOSE)

        value_continuation_pattern = re.compile(r"^(?!\s)(?!.*\s$)(?P<value>.+)$")
        
        for line in lines:
            line = line.strip()
            if not line:
                continue

            # Detect start of fields section
            if not in_field_section:
                if "generated" in line.lower():
                    in_field_section = True
                    continue
                header_lines.append(line)
                continue

#             # Parse field-value pairs
#             #print(line)
#             match = field_pattern.match(line)
#             #cont_match = value_continuation_pattern.match(line)
            
#             if match:
#                 current_field = match.group("field").strip()
# #                 value = match.group("value").strip()
#                 #print(current_field)
#                 value = ""
#                 field_value_map.setdefault(current_field, []).append(value)
                
#         final_fields = [{"field_name": k} for k, v in field_value_map.items()]
        
        

        header_text = " ".join([j for j in header_lines if "form" in j.lower() or "folder" in j.lower()]).strip()
        
        if final_fields and header_text:
            
            return {
                
                "template_name": os.path.basename(file_path),
                "path": file_path,
                "id": str(uuid4()),
                "source_data": {
                    "assessments": header_text,
                    "fields": final_fields,
                },
            }

        return None


In [469]:
#necessary imports 
from datetime import datetime
import traceback
import dataikuapi
import io

import base64
import uuid
# import from GLOBAL SHARED CODE
from utils import connection 
# imports from library 
import re
from utilities.variables import RD_PROJECT_NAME
from utilities.variables import SECRET_NAME , TOKEN_KEY
from utilities.logging_config import logging
# from soa_extraction.crf_extraction import HistoricalCRF
# import HistoricalCRF
import traceback
import os


def extract_crf(file_path):
    """
    Input Args:
        file_path: str
        
    Response:
        result : dict
    """
    try:
        logging.info("Intializing client for file_upload function ")
       
        DATAIKU_HOST , API_SECRET_KEY  = connection.get_dataiku_host_and_api_key(RD_PROJECT_NAME,SECRET_NAME,TOKEN_KEY)
        
        client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
        project = client.get_project(RD_PROJECT_NAME)
        proj_vars = project.get_variables()["local"]
        
        snowflake_conn = proj_vars.get("snowflake_connection_string")
        
        obj = HistoricalCRF(client , project)
        final_list = []
        response,response_oid = obj.historical_mapping(file_path,"e3afcae7-6133-4737-b641-81d928469a31")
        
        from collections import defaultdict
        import copy
        def merge_sections_per_file(results,field_name):
            merged = defaultdict(dict)

            for item in results:
                file_path = item["path"]
                assessment = item["source_data"]["assessments"]

                if assessment not in merged[file_path]:
                    merged[file_path][assessment] = copy.deepcopy(item)
                else:
                    # merge fields for same file + same assessment
                    merged[file_path][assessment]["source_data"][field_name] += item["source_data"][field_name]


            # flatten structure
            final = []
            for file_assessments in merged.values():
                final.extend(file_assessments.values())

            return final

        ans = merge_sections_per_file(response,"fields")
        ans_oid = merge_sections_per_file(response_oid,"fields_oid")
        
#         print(ans_oid)
        merged_list = []
        for d1 in ans:
            matched = False

            for d2 in ans_oid:
                if (
                    d1["template_name"] == d2["template_name"]
                    and d1["source_data"]["assessments"] == d2["source_data"]["assessments"]
                ):
                    
                    fields = d1["source_data"]["fields"]
                    fields_oid = d2["source_data"]["fields_oid"]

                    
                    for i, field in enumerate(fields):
                        found_oids_name = None
                        for each_oids in fields_oid:
                            if int(each_oids["field_number"]) == int(field["field_number"]):
                                found_oids_name = each_oids["field_name"]
                        
                        field["field_oid"] = found_oids_name

                    merged_list.append(d1)
                    matched = True
                    break  # stop checking after first match

            if not matched:
                # keep original entry unchanged
                merged_list.append(d1)

        # --- Result ---
#         import pprint
#         pprint.pprint(merged_list)
        
        for resp in merged_list:
            form_name = resp['source_data']['assessments']
            match = re.search(r'Form[:\s]*(.*)', form_name)
            if match:
                form_name = match.group(1)
            for field in resp['source_data']['fields']:
                
                field_name = field['field_name']
                field_oid = field['field_oid']
                #field_oid = field["field_oid"]
                final_list.append({
                    "form_name": form_name,
                    "field_name": re.split(r'\t|\s{2,}', field_name.strip())[0],
                    "field_oid" : field_oid
                    #"field_oid" : field_oid
                })

        # ✅ Deduplicate based on both form_name and field_name
        seen = set()
        deduped_list = []
        for item in final_list:
            key = (item["form_name"].strip().lower(), item["field_name"].strip().lower())
            if key not in seen:
                seen.add(key)
                deduped_list.append(item)

        return {
            "result": deduped_list
        }

    except:
        t = traceback.format_exc()
        logging.error(f"Error caused due to {t}")
        return {"message": f"Error caused due to {t}"}


In [470]:
output_crfs = extract_crf('/Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1).pdf')
output_crfs

[{'field_number': 1, 'field_name': 'SITENUM'}, {'field_number': 2, 'field_name': 'SUBJID'}, {'field_number': 3, 'field_name': 'SUBJNAME'}]
[{'field_number': 1, 'field_name': 'SVSTDAT'}]
[{'field_number': 1, 'field_name': 'DSYN'}, {'field_number': 2, 'field_name': 'DSSTDAT'}, {'field_number': 3, 'field_name': 'DSTIM'}, {'field_number': 4, 'field_name': 'DSDTTM'}, {'field_number': 5, 'field_name': 'DSSAMND'}, {'field_number': 6, 'field_name': 'DSDECOD_3'}, {'field_number': 7, 'field_name': 'PROTOCOL'}]
[{'field_number': 1, 'field_name': 'DOB'}, {'field_number': 2, 'field_name': 'AGE'}, {'field_number': 3, 'field_name': 'SEX'}, {'field_number': 4, 'field_name': 'GENDER'}, {'field_number': 5, 'field_name': 'CHILDBR'}, {'field_number': 6, 'field_name': 'CHILDBRN'}, {'field_number': 7, 'field_name': 'CHILDBROT'}, {'field_number': 8, 'field_name': 'REPPOT'}, {'field_number': 10, 'field_name': 'IMPLANT'}, {'field_number': 11, 'field_name': 'INTRADEV'}, {'field_number': 12, 'field_name': 'INTRA

[{'field_number': 85, 'field_name': 'CSS0423B'}, {'field_number': 86, 'field_name': 'CSS0423C'}]
[{'field_number': 1, 'field_name': 'SVSTDAT'}]
[{'field_number': 1, 'field_name': 'IEYN'}, {'field_number': 2, 'field_name': 'IECAT'}, {'field_number': 3, 'field_name': 'IETESTCD'}]
[{'field_number': 1, 'field_name': 'PEYN'}, {'field_number': 2, 'field_name': 'PESPEC'}, {'field_number': 3, 'field_name': 'PEDAT'}, {'field_number': 4, 'field_name': 'PETIM'}, {'field_number': 5, 'field_name': 'PEDTTM'}, {'field_number': 6, 'field_name': 'PETEST'}, {'field_number': 7, 'field_name': 'PEOTHSP'}, {'field_number': 8, 'field_name': 'PESTAT'}, {'field_number': 9, 'field_name': 'PEREASND'}, {'field_number': 10, 'field_name': 'PEORRES'}, {'field_number': 11, 'field_name': 'PEDESC'}]
[{'field_number': 1, 'field_name': 'VSYN'}, {'field_number': 2, 'field_name': 'VSREASND'}, {'field_number': 3, 'field_name': 'VSDAT'}, {'field_number': 4, 'field_name': 'VSTIM'}, {'field_number': 5, 'field_name': 'VSDTTM'},

[{'field_number': 1, 'field_name': 'EXFAST'}, {'field_number': 2, 'field_name': 'EXYN'}, {'field_number': 3, 'field_name': 'EXMC'}, {'field_number': 4, 'field_name': 'EXTPT'}, {'field_number': 5, 'field_name': 'EXTRT'}, {'field_number': 6, 'field_name': 'EXADJYN'}, {'field_number': 7, 'field_name': 'EXADJ'}, {'field_number': 8, 'field_name': 'EXDOSFRM'}, {'field_number': 9, 'field_name': 'EXDOSESTR5'}, {'field_number': 10, 'field_name': 'EXDOSSTUN'}, {'field_number': 11, 'field_name': 'EXUNITS'}, {'field_number': 12, 'field_name': 'EXDOSE'}, {'field_number': 13, 'field_name': 'EXDOSU'}, {'field_number': 14, 'field_name': 'EXDKYN'}, {'field_number': 15, 'field_name': 'EXVOLYN'}, {'field_number': 16, 'field_name': 'EXROUTE'}, {'field_number': 17, 'field_name': 'EXSTDAT'}, {'field_number': 18, 'field_name': 'EXSTTIM'}]
[{'field_number': 19, 'field_name': 'EXSTDTTM'}]
[{'field_number': 1, 'field_name': 'VSTPT'}, {'field_number': 2, 'field_name': 'VSTPTNUM'}, {'field_number': 3, 'field_name

[{'field_number': 1, 'field_name': 'EXFAST'}, {'field_number': 2, 'field_name': 'EXYN'}, {'field_number': 3, 'field_name': 'EXMC'}, {'field_number': 4, 'field_name': 'EXTPT'}, {'field_number': 5, 'field_name': 'EXTRT'}, {'field_number': 6, 'field_name': 'EXADJYN'}, {'field_number': 7, 'field_name': 'EXADJ'}, {'field_number': 8, 'field_name': 'EXDOSFRM'}, {'field_number': 9, 'field_name': 'EXDOSESTR5'}, {'field_number': 10, 'field_name': 'EXDOSSTUN'}, {'field_number': 11, 'field_name': 'EXUNITS'}, {'field_number': 12, 'field_name': 'EXDOSE'}, {'field_number': 13, 'field_name': 'EXDOSU'}, {'field_number': 14, 'field_name': 'EXDKYN'}, {'field_number': 15, 'field_name': 'EXVOLYN'}, {'field_number': 16, 'field_name': 'EXROUTE'}, {'field_number': 17, 'field_name': 'EXSTDAT'}, {'field_number': 18, 'field_name': 'EXSTTIM'}]
[{'field_number': 19, 'field_name': 'EXSTDTTM'}]
[{'field_number': 1, 'field_name': 'EXFAST'}, {'field_number': 2, 'field_name': 'EXYN'}, {'field_number': 3, 'field_name': 

[{'field_number': 1, 'field_name': 'LBCAT'}, {'field_number': 2, 'field_name': 'LBSPEC'}, {'field_number': 3, 'field_name': 'LBYN'}, {'field_number': 4, 'field_name': 'LBREASND'}, {'field_number': 5, 'field_name': 'LBDAT'}, {'field_number': 6, 'field_name': 'LBTIM'}, {'field_number': 7, 'field_name': 'LBDTTM'}, {'field_number': 8, 'field_name': 'LBCLSIG'}, {'field_number': 9, 'field_name': 'LBCSTST1'}, {'field_number': 10, 'field_name': 'LBCSTST2'}, {'field_number': 11, 'field_name': 'LBCSTST3'}, {'field_number': 12, 'field_name': 'LBOTHSPE'}, {'field_number': 13, 'field_name': 'LBRPT'}]
[{'field_number': 1, 'field_name': 'LBCAT'}, {'field_number': 2, 'field_name': 'LBSPEC_4'}, {'field_number': 3, 'field_name': 'LBYN'}, {'field_number': 4, 'field_name': 'LBREASND'}, {'field_number': 5, 'field_name': 'LBDAT'}, {'field_number': 6, 'field_name': 'LBTIM'}, {'field_number': 7, 'field_name': 'LBDTTM'}, {'field_number': 8, 'field_name': 'LBRPT'}]
[{'field_number': 1, 'field_name': 'PCSPEC'}, 

...

[{'field_number': 1, 'field_name': 'EXFAST'}, {'field_number': 2, 'field_name': 'EXYN'}, {'field_number': 3, 'field_name': 'EXMC'}, {'field_number': 4, 'field_name': 'EXTPT'}, {'field_number': 5, 'field_name': 'EXTRT'}, {'field_number': 6, 'field_name': 'EXADJYN'}, {'field_number': 7, 'field_name': 'EXADJ'}, {'field_number': 8, 'field_name': 'EXDOSFRM'}, {'field_number': 9, 'field_name': 'EXDOSESTR5'}, {'field_number': 10, 'field_name': 'EXDOSSTUN'}, {'field_number': 11, 'field_name': 'EXUNITS'}, {'field_number': 12, 'field_name': 'EXDOSE'}, {'field_number': 13, 'field_name': 'EXDOSU'}, {'field_number': 14, 'field_name': 'EXDKYN'}, {'field_number': 15, 'field_name': 'EXVOLYN'}, {'field_number': 16, 'field_name': 'EXROUTE'}, {'field_number': 17, 'field_name': 'EXSTDAT'}, {'field_number': 18, 'field_name': 'EXSTTIM'}]
[{'field_number': 19, 'field_name': 'EXSTDTTM'}]
[{'field_number': 1, 'field_name': 'VSTPT'}, {'field_number': 2, 'field_name': 'VSTPTNUM'}, {'field_number': 3, 'field_name

[{'field_number': 1, 'field_name': 'PCSPEC'}, {'field_number': 2, 'field_name': 'PCTPT'}, {'field_number': 3, 'field_name': 'PCTPTNUM'}, {'field_number': 4, 'field_name': 'PCTPTREF'}, {'field_number': 5, 'field_name': 'PCYN'}, {'field_number': 6, 'field_name': 'PCREASND'}, {'field_number': 7, 'field_name': 'PCDAT'}, {'field_number': 8, 'field_name': 'PCTIM'}, {'field_number': 9, 'field_name': 'PCDTTM'}, {'field_number': 10, 'field_name': 'PCCOM'}, {'field_number': 11, 'field_name': 'PCACCESS1'}, {'field_number': 12, 'field_name': 'PCACCESS2'}]
[{'field_number': 1, 'field_name': 'SVSTDAT'}]
[{'field_number': 1, 'field_name': 'PCSPEC'}, {'field_number': 2, 'field_name': 'PCTPT'}, {'field_number': 3, 'field_name': 'PCTPTNUM'}, {'field_number': 4, 'field_name': 'PCTPTREF'}, {'field_number': 5, 'field_name': 'PCYN'}, {'field_number': 6, 'field_name': 'PCREASND'}, {'field_number': 7, 'field_name': 'PCDAT'}, {'field_number': 8, 'field_name': 'PCTIM'}, {'field_number': 9, 'field_name': 'PCDTTM

[{'field_number': 1, 'field_name': 'EGYN'}, {'field_number': 2, 'field_name': 'EGREASND'}, {'field_number': 3, 'field_name': 'EGDAT'}, {'field_number': 4, 'field_name': 'EGTIM'}, {'field_number': 5, 'field_name': 'EGSTDTTM'}, {'field_number': 6, 'field_name': 'EGPOS'}, {'field_number': 7, 'field_name': 'HR'}, {'field_number': 8, 'field_name': 'PR'}, {'field_number': 9, 'field_name': 'QRS'}, {'field_number': 10, 'field_name': 'QT'}, {'field_number': 11, 'field_name': 'QTCF'}, {'field_number': 12, 'field_name': 'RR'}, {'field_number': 13, 'field_name': 'INTP'}, {'field_number': 14, 'field_name': 'EGCOM'}, {'field_number': 15, 'field_name': 'EGRPT'}]
[{'field_number': 1, 'field_name': 'LBCAT'}, {'field_number': 2, 'field_name': 'LBSPEC'}, {'field_number': 3, 'field_name': 'LBYN'}, {'field_number': 4, 'field_name': 'LBREASND'}, {'field_number': 5, 'field_name': 'LBDAT'}, {'field_number': 6, 'field_name': 'LBTIM'}, {'field_number': 7, 'field_name': 'LBDTTM'}, {'field_number': 8, 'field_name

[{'field_number': 1, 'field_name': 'VSREPEAT'}, {'field_number': 2, 'field_name': 'VSDAT'}, {'field_number': 3, 'field_name': 'VSTIM'}, {'field_number': 4, 'field_name': 'VSDTTM'}, {'field_number': 5, 'field_name': 'VSPOS'}, {'field_number': 6, 'field_name': 'SYSBP'}, {'field_number': 7, 'field_name': 'DIABP'}, {'field_number': 8, 'field_name': 'PULSE'}, {'field_number': 9, 'field_name': 'TEMP'}, {'field_number': 10, 'field_name': 'VSCLSIG'}, {'field_number': 11, 'field_name': 'VSTEST1'}, {'field_number': 12, 'field_name': 'VSTEST2'}, {'field_number': 13, 'field_name': 'VSTEST3'}, {'field_number': 14, 'field_name': 'VSRPT'}]
[{'field_number': 1, 'field_name': 'VSREPEAT_'}, {'field_number': 1, 'field_name': 'U'}, {'field_number': 2, 'field_name': 'VSDAT'}, {'field_number': 3, 'field_name': 'VSTIM'}, {'field_number': 4, 'field_name': 'VSDTTM'}, {'field_number': 5, 'field_name': 'WEIGHT'}]
[{'field_number': 1, 'field_name': 'EGREPEAT'}, {'field_number': 2, 'field_name': 'EGDAT'}, {'field_

{'result': [{'form_name': 'Enrollment',
   'field_name': 'Site ID',
   'field_oid': 'SITENUM'},
  {'form_name': 'Enrollment',
   'field_name': 'Participant ID',
   'field_oid': 'SUBJID'},
  {'form_name': 'Enrollment',
   'field_name': 'Participant Number (Derived)',
   'field_oid': 'SUBJNAME'},
  {'form_name': 'Date of Visit',
   'field_name': 'Visit date',
   'field_oid': 'SVSTDAT'},
  {'form_name': 'Informed Consent',
   'field_name': 'Informed consent obtained?',
   'field_oid': 'DSYN'},
  {'form_name': 'Informed Consent',
   'field_name': 'Informed consent date',
   'field_oid': 'DSSTDAT'},
  {'form_name': 'Informed Consent',
   'field_name': 'Informed consent time',
   'field_oid': 'DSTIM'},
  {'form_name': 'Informed Consent',
   'field_name': 'Derived date',
   'field_oid': 'DSDTTM'},
  {'form_name': 'Informed Consent',
   'field_name': 'Informed consent version number',
   'field_oid': 'DSSAMND'},
  {'form_name': 'Informed Consent',
   'field_name': 'Standardized disposition ter

In [475]:
import pandas as pd
df = pd.DataFrame(output_crfs["result"])
display(df.head(40))

,form_name,field_name,field_oid
0,Enrollment,Site ID,SITENUM
1,Enrollment,Participant ID,SUBJID
2,Enrollment,Participant Number (Derived),SUBJNAME
3,Date of Visit,Visit date,SVSTDAT
4,Informed Consent,Informed consent obtained?,DSYN
5,Informed Consent,Informed consent date,DSSTDAT
6,Informed Consent,Informed consent time,DSTIM
7,Informed Consent,Derived date,DSDTTM
8,Informed Consent,Informed consent version number,DSSAMND
9,Informed Consent,Standardized disposition term,DSDECOD_3


In [478]:
import pandas as pd
import io



# Convert DataFrame to CSV bytes
buffer = io.StringIO()
df.to_csv(buffer, index=False)
csv_bytes = buffer.getvalue().encode("utf-8")  # Convert to bytes for upload

# Initialize your CRF object
obj = HistoricalCRF(client , proj)

# Upload CSV to Dataiku managed folder
obj.input_folder.put_file("/extracted_fields_with_oids.csv", io.BytesIO(csv_bytes))
print("✅ CSV uploaded successfully to managed folder as /output.csv")

# (Optional) Save locally too
with open("output.csv", "wb") as f:
    f.write(csv_bytes)
print("✅ CSV also saved locally as output.csv")


✅ CSV uploaded successfully to managed folder as /output.csv
✅ CSV also saved locally as output.csv


# 🔍 Hybrid Search over OpenSearch with LLM Fallback

This section performs a **hybrid search** over an OpenSearch index to find the most relevant **form-field mappings** for extracted CRF data.  
If the similarity score is below a defined threshold, it falls back to an **LLM-based generator** to produce deterministic specifications.

---

## 🛠 Workflow Overview

### **1. OpenSearch Setup**
- Initializes an `OpensearchUtil` client using the connected Dataiku project.
- Retrieves the OpenSearch index name from project variables and replaces any `${projectKey}` placeholders with the actual project key.

---

### **2. Iterating over Extracted CRF Data**
- Loops through each `(form_name, field_name)` pair from `sub_data` (first 101 rows of digitized CRF data).
- Generates **vector embeddings** for both `form_name` and `field_name` using the default embedding model defined in project variables.

---

### **3. Hybrid Query Construction**
- Builds an OpenSearch **kNN (vector) search query**:
  - Matches on `form_name_vector` and `form_field_value_vector`.
  - Retrieves the top 10 nearest neighbors for each vector.

---

### **4. Search Execution & Result Scoring**
- Executes the query and collects candidate hits.
- For each hit:
  - Computes **cosine similarity** between field vectors.
  - Computes **fuzzy text similarity** using `rapidfuzz`.
  - Combines them into a **hybrid score** using weighted average:  
    \[
    \text{final_score} = 0.7 \times \text{cosine_similarity} + 0.3 \times \text{text_similarity}
    \]
- Selects the hit with the highest combined score as the **best match**.

---

### **5. Threshold-Based LLM Fallback**
- If the best match's **normalized OpenSearch score** `< 0.64` →  
  Calls `make_llm_call(form_name, field_name)` to generate an **LLM-based edit-check specification**.
- Even if the top hit passes the initial threshold, it **re-checks field-level similarity**:
  - If the recomputed field-level similarity `< 0.64`, it again calls the LLM.
  - Otherwise, it keeps the OpenSearch result as-is.

---

### **6. Enriching Results**
- For LLM-generated results:
  - Adds metadata (ecs_id, form_id, original form & field names, similarity scores).
  - Marks the source as **"LLM Generated"** for traceability.
- For OpenSearch results:
  - Updates the `form_field_value` to the best-matched value and stores it.

---

### **7. Final Output**
- All results (both OpenSearch and LLM-generated) are collected into `output_list`.
- Converted to a Pandas DataFrame (`final_df2`) for further processing or review.
- Execution time is printed for performance monitoring.

---

## ✅ **Key Points**
- **Hybrid similarity** = Cosine similarity (semantic) + Fuzzy match (string-based).
- **LLM Fallback**:
  - Triggered if similarity < **0.64** (either overall hit score or field-level score).
  - Ensures robust, deterministic edit-check specification generation even when OpenSearch recall is weak.
- **Traceability**: All outputs include metadata (`source`, `score`, `field_score`) to distinguish model-generated vs. search-retrieved results.

---




In [34]:
# Single search over OpenSearch index with hybrid similarity (vector + fuzzy)
import time
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz  # for fuzzy text similarity

start = time.time()

# Initialize OpenSearch client
opensearch_client = OpensearchUtil(client, proj)
client_os = opensearch_client.opensearch_client



{'type': 'ElasticSearch', 'params': {'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'username': 'genai-admin', 'password': 'qPk7Jf5vcyXDS!**332gXTvSfmcauvr9', 'port': 443, 'ssl': True, 'trustAnySSLCertificate': True, 'dialect': 'ES_7', 'dkuProperties': [], 'namingRule': {'indexNameDatasetNamePrefix': '${projectKey}_'}, 'authType': 'PASSWORD', 'oauth': {'refreshTokenRotation': False}, 'aws': {'service': 'OPENSEARCH_SERVERLESS', 'credentialsMode': 'KEYPAIR', 'customAWSCredentialsProviderParams': []}}, 'credentialsMode': 'GLOBAL', 'proxySettingsAsString': ''}
opensearch <OpenSearch([{'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'port': 443}])>


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/opensearchpy/connection/http_urllib3.py:214: UserWarning: Connecting to https://aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com:443 using SSL with verify_certs=False is insecure.
  warnings.warn(


In [74]:
import time
import json
import pandas as pd
from rapidfuzz import fuzz, process


class CRFFuzzyMatcher:
    def __init__(self, opensearch_client, llm_fallback=None, score_threshold=0.40):
        """
        :param opensearch_client: Initialized OpenSearch client
        :param index_name: Name of the OpenSearch index
        :param llm_fallback: Function(form_name, field_name) -> dict | None
        :param score_threshold: Threshold for LLM fallback (0–1)
        """
        self.opensearch_client = OpensearchUtil(client, proj).opensearch_client
        #self.opensearch_client = opensearch_client
        self.index_name = proj.get_variables()['local'].get('ecs_opensearch')
        dataiku_project_var = "${projectKey}"
        if dataiku_project_var in self.index_name:
            # Replace projectKey placeholder with actual project key
            self.index_name = self.index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()

        
        self.llm_fallback = llm_fallback
        self.score_threshold = score_threshold

    #from rapidfuzz import process, fuzz
    
    
    def get_all_standard_forms(self):
        """
        Fetch all unique standard form names from OpenSearch once.
        """
        query_all_forms = {
            "size": 1000,  # adjust if you have more than 1000 forms
            "_source": ["form_name"],
            "query": {
                "bool": {
                    "must": {"match_all": {}},
                    "filter": [
                        {"term": {"source.keyword": "Standard"}}
                    ]
                }
            }
        }

        res = self.opensearch_client.search(index=self.index_name, body=json.dumps(query_all_forms))
        hits = res.get("hits", {}).get("hits", [])
        return [hit["_source"]["form_name"] for hit in hits]


    def get_standard_crf(self, form_name,all_form_names):
        """
        Fetch CRF standards using fuzzy matching for form_name first.
        Then query OpenSearch for the matched standard form.
        """
        if not form_name:
            return pd.DataFrame([])

        form_name_clean = form_name.strip()

        # Step 1: Get list of all unique standard form names from OpenSearch
        

        if not all_form_names:
            return pd.DataFrame([])

        # Step 2: Fuzzy match input form_name to standard form
        matched_form, score, _ = process.extractOne(
            form_name_clean, all_form_names, scorer=fuzz.partial_ratio
        )

        
        
        if score < 70:
            return pd.DataFrame([])
        # Step 3: Query OpenSearch for the matched standard form
        
        print(f"This is the matched form name : ",{matched_form})
        query = {
            "query": {
                "term": {"form_name.keyword": matched_form}
            }
        }
        
        query = {
            "size": 10000,  # adjust if you have more than 1000 forms
#             "_source": ["form_name"],
            "query": {
                "bool": {
                    "must": {"match_all": {}},
                    "filter": [
                        {"term": {"form_name.keyword": matched_form}}
                    ]
                }
            }
        }

        res_std = self.opensearch_client.search(index=self.index_name, body=json.dumps(query))
        hits_std = res_std.get("hits", {}).get("hits", [])

        if not hits_std:
            return pd.DataFrame([])

        results = [hit["_source"] for hit in hits_std]

        return pd.DataFrame([
            {
                "validation_id": item.get("validation_id"),
                "ecs_id": item.get("ecs_id"),
                "form_id": item.get("form_id"),
                "form_name": item.get("form_name"),
                "form_field_value": item.get("form_field_value"),
                "validation_logic": item.get("validation_logic"),
                "reasoning": item.get("reasoning"),
                "action": item.get("action"),
                "source": item.get("source"),
                "action_details": item.get("action_details"),
            }
            for item in results
        ])
    
    def get_all_historical_forms(self):
        """
        Fetch all unique standard form names from OpenSearch once.
        """
        query_all_forms = {
            "size": 1000,  # adjust if you have more than 1000 forms
            "_source": ["form_name"],
            "query": {
                "bool": {
                    "must": {"match_all": {}},
                    "filter": [
                        {"term": {"source.keyword": "Historic"}}
                    ]
                }
            }
        }

        res = self.opensearch_client.search(index=self.index_name, body=json.dumps(query_all_forms))
        hits = res.get("hits", {}).get("hits", [])
        return [hit["_source"]["form_name"] for hit in hits]

    def fuzzy_match_fields(self, sub_data):
        """
        Run fuzzy matching for each field in sub_data.
        1. Try matching against Standard CRFs.
        2. Try remaining unmatched rows against Historical CRFs.
        3. Use LLM fallback for anything unmatched or below threshold.
        """
        output_list = []

        all_standard_forms = self.get_all_standard_forms()
        all_historical_forms = self.get_all_historical_forms()
        print(set(all_historical_forms))
        for row in sub_data:
            print(row)
            form_name = row['form_name']
            field_name = row.get('field_name')

            if not form_name or not field_name:
                continue

            print(f"🔍 Matching: {form_name} — {field_name}")

            matched_row = None

            # ---------- STEP 1: Try Standard ----------
            reference_df = self.get_standard_crf(form_name, all_standard_forms)
            print(f"length of standard forms for particula form : {form_name} ", len(reference_df))
            if not reference_df.empty:
                choices_df = reference_df['form_field_value'].dropna().reset_index()  
                # choices_df columns: ['index', 'form_field_value'] where 'index' is the original index

                reference_choices = choices_df['form_field_value'].tolist()
                
                
                print(f"length of reference choices for particula form : {form_name} ", len(reference_choices))
                if reference_choices:
                    result = process.extractOne(field_name, reference_choices, scorer=fuzz.token_sort_ratio)
                    
                    if result:
                        best_match, score, match_index = result
                        original_df_index = choices_df.loc[match_index, 'index']  # recover original index
                        matched_row = reference_df.loc[original_df_index].to_dict()
                        print(f"❤️ Matched Index : ",{match_index},result)
                        #matched_row = reference_df.iloc[match_index].to_dict()
                        print(f"✅ Form name : {form_name}, Feild Name : {field_name} Matched_Index : {matched_row}, score: {score}")
                        
                        matched_row.update({
                            "original_form_name": form_name,
                            "original_field": field_name,
                            "score": score / 100.0,
                            "source": "Fuzzy Match (Standard)"
                        })

            # ---------- STEP 2: Try Historical ----------
            if matched_row is None:
                reference_df = self.get_standard_crf(form_name, all_historical_forms)
                print(f"length of standard forms for particula form : {form_name} ", len(reference_df))
                if not reference_df.empty:
                    print(f"❌ Not from Standard for {form_name} going to Historical")
                    choices_df = reference_df['form_field_value'].dropna().reset_index()  
                    # choices_df columns: ['index', 'form_field_value'] where 'index' is the original index

                    reference_choices = choices_df['form_field_value'].tolist()
                    
                    print(f"length of reference choices for particula form : {form_name} ", len(reference_choices))
                    
                    if reference_choices:
                        result = process.extractOne(field_name, reference_choices, scorer=fuzz.partial_ratio)
                        if result:
                            best_match, score, match_index = result
                            original_df_index = choices_df.loc[match_index, 'index']  # recover original index
                            matched_row = reference_df.loc[original_df_index].to_dict()
                            print(f"❤️ Matched Index : ",{match_index},result)
                            #matched_row = reference_df.iloc[match_index].to_dict()
                            
                            if score >= self.score_threshold*100:
                                print(f"✅ Form name : {form_name}, Feild Name : {field_name} Matched_Index : {matched_row}, score : {score}")
                            matched_row.update({
                                "original_form_name": form_name,
                                "original_field": field_name,
                                "score": score / 100.0,
                                "source": "Fuzzy Match (Historical)"
                            })

            # ---------- STEP 3: LLM fallback ----------
            if matched_row is None or matched_row["score"] < self.score_threshold:
                if self.llm_fallback:
                    llm_output = self.llm_fallback(form_name, field_name) or {}
                    llm_output.update({
                        "ecs_id": matched_row.get("ecs_id") if matched_row else None,
                        "form_id": matched_row.get("form_id") if matched_row else None,
                        "original_form_name": form_name,
                        "original_field": field_name,
                        "score": matched_row.get("score") if matched_row else None,
                        "source": "LLM Generated"
                    })
                    print(f"❌ Not found in Historical for {form_name} going to LLM generated")
                    matched_row = llm_output

#             # ---------- STEP 4: Handle Not Found ----------
#             if matched_row is None:
#                 matched_row = {
#                     "ecs_id": None,
#                     "form_id": None,
#                     "score": None,
#                     "source": "Not Found",
#                     "original_form_name": form_name,
#                     "original_field": field_name,
#                     "form_name": None,
#                     "form_field_value": None,
#                     "validation_logic": None,
#                     "reasoning": None,
#                     "action": None,
#                     "action_details": None,
#                 }

            output_list.append(matched_row)

        return pd.DataFrame(output_list)

# ----------------- Usage -----------------

def make_llm_call(form_name, field_name):
    """
    Dummy LLM fallback. Replace with real implementation.
    Should return dict with keys like {"form_field_value": "..."}.
    """
    return {"form_field_value": f"Not found"}


if __name__ == "__main__":
    start = time.time()

    # Example: initialize
    # opensearch_client = <your existing client>
    # index_name = proj.get_variables()['local'].get('ecs_opensearch')

    matcher = CRFFuzzyMatcher(
        opensearch_client=opensearch_client,
        
        llm_fallback=make_llm_call,   # set None if no LLM fallback
        score_threshold=0.50
    )

    # Run fuzzy matching (queries OpenSearch per form_name)
    final_df2 = matcher.fuzzy_match_fields(output_crfs["result"])
    display(final_df2)
    end = time.time()
    print(f"Search completed in {end - start:.2f} seconds")
    

{'type': 'ElasticSearch', 'params': {'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'username': 'genai-admin', 'password': 'qPk7Jf5vcyXDS!**332gXTvSfmcauvr9', 'port': 443, 'ssl': True, 'trustAnySSLCertificate': True, 'dialect': 'ES_7', 'dkuProperties': [], 'namingRule': {'indexNameDatasetNamePrefix': '${projectKey}_'}, 'authType': 'PASSWORD', 'oauth': {'refreshTokenRotation': False}, 'aws': {'service': 'OPENSEARCH_SERVERLESS', 'credentialsMode': 'KEYPAIR', 'customAWSCredentialsProviderParams': []}}, 'credentialsMode': 'GLOBAL', 'proxySettingsAsString': ''}
opensearch <OpenSearch([{'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'port': 443}])>


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/opensearchpy/connection/http_urllib3.py:214: UserWarning: Connecting to https://aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com:443 using SSL with verify_certs=False is insecure.
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verific

{'Document Questions and Responses - Paper Consent', 'Study Drug Transfer Log - Combined', 'Blood Collection - PK SEP/PD Creatine', 'Structured Interview Manual for Young Mania Rating Scale (YMRS) Last 7 Days', 'Pupillometry - Single Time Point', 'Pupillometry - Multi Time Point - Part 1_V2', 'Visit Date', 'Screening Status', 'Challenging Experience Questionnaire (CEQ)', 'PK Sampling - Single Time Point', 'Pupillometry - Multi Time Point - Part 1', 'Physical Examination', 'Cental Lab', 'Clinical Global Impression - Severity of Illness (CGI-S)', 'FSH Test', 'ECG (RScribe) Admission', 'Urine PK Sampling - Day -1', 'Sleep EEG - In Clinic', 'ECG (RScribe) Screening', 'Dosing Session Release Checklist', 'Inclusion/Exclusion Criteria', 'Serology', 'PK Sampling - Multi Time Point - Part 2', 'Prior and Concomitant Medication Log', '12-Lead ECG Triplicate Log', 'Height & Weight & BMI', 'Body Measurements', 'Custom Function', 'Pregnancy Test', 'Fluid Biomarkers', 'Physical Examination - Abbrevia

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

length of standard forms for particula form : Informed Consent  32
length of reference choices for particula form : Informed Consent  12
❤️ Matched Index :  {8} ('Signed by', 38.095238095238095, 8)
✅ Form name : Informed Consent, Feild Name : Derived date Matched_Index : {'validation_id': 'MVAL_DS_IC003', 'ecs_id': '74b75f61-dc2b-4150-8a85-f49e12580e77', 'form_id': 'c3196559-2e4e-4c3e-bd98-afcd706447a2', 'form_name': 'Informed Consent', 'form_field_value': 'Signed by', 'validation_logic': '(DS_IC.IFCOCCUR == "Yes") then (DS_IC.QVAL_IFCSIGN is enterable)', 'reasoning': 'If informed consent was signed then Signed by is a valid field for entry', 'action': 'DS_IC.DSSTTIM_IC is enterable', 'source': 'Standard', 'action_details': '<n/a>'}, score: 38.095238095238095
❌ Not found in Historical for Informed Consent going to LLM generated
{'form_name': 'Informed Consent', 'field_name': 'Informed consent version number'}
🔍 Matching: Informed Consent — Informed consent version number
This is the ma

...

length of standard forms for particula form : Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit (Uns)  13
❌ Not from Standard for Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit (Uns) going to Historical
length of reference choices for particula form : Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit (Uns)  0
❌ Not found in Historical for Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit (Uns) going to LLM generated
{'form_name': 'Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit (Uns)', 'field_name': 'Actual Attempt:'}
🔍 Matching: Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit (Uns) — Actual Attempt:
length of standard forms for particula form : Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit (Uns)  0
This is the matched form name :  {'Columbia-Suicide Severity Rating Scale (C-SSRS) Since Last Visit'}
length of standard forms for particula form : Columbia Suicide Sev

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

length of standard forms for particula form : Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit (Uns)  13
❌ Not from Standard for Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit (Uns) going to Historical
length of reference choices for particula form : Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit (Uns)  0
❌ Not found in Historical for Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit (Uns) going to LLM generated
{'form_name': 'Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit (Uns)', 'field_name': 'Completed Suicide:'}
🔍 Matching: Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit (Uns) — Completed Suicide:
length of standard forms for particula form : Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit (Uns)  0
This is the matched form name :  {'Columbia-Suicide Severity Rating Scale (C-SSRS) Since Last Visit'}
length of standard forms for particula form : Columbia Suici

,form_field_value,ecs_id,form_id,original_form_name,original_field,score,source,validation_id,form_name,validation_logic,reasoning,action,action_details
0,Not found,None,None,Enrollment,Site ID,NaN,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN
1,Not found,None,None,Enrollment,Participant ID,NaN,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN
2,Not found,None,None,Enrollment,Participant Number (Derived),NaN,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN
3,Total number of aborted attempts.,d6f6eb1b-ee4a-4975-af12-c2731fb43b3b,3e5a80e4-c245-440f-8496-185e690ba436,Date of Visit,Visit date,0.500000,Fuzzy Match (Historical),None,C-SSRS - Since Last Visit,Query if the total number of aborted attempts ...,None,,
4,Was informed consent obtained?,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent obtained?,0.642857,Fuzzy Match (Standard),MVAL_DS_IC006,Informed Consent,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>
...,...,...,...,...,...,...,...,...,...,...,...,...,...
890,Not found,None,None,Columbia Suicide Severity Rating Scale (C-SSRS...,Preparatory Acts or Behavior:,NaN,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN
891,Not found,None,None,Columbia Suicide Severity Rating Scale (C-SSRS...,Suicidal behavior was present during the asses...,NaN,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN
892,Not found,None,None,Columbia Suicide Severity Rating Scale (C-SSRS...,Completed Suicide:,NaN,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN
893,Not found,None,None,Columbia Suicide Severity Rating Scale (C-SSRS...,Most Lethal Attempt Date (dd MMM yyyy),NaN,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN


Search completed in 27.29 seconds


In [75]:
import pandas as pd
import io

# Sample DataFrame
df = final_df2

# Convert DataFrame to CSV bytes
buffer = io.StringIO()
df.to_csv(buffer, index=False)
csv_bytes = buffer.getvalue().encode("utf-8")  # Convert to bytes for upload

# Initialize your CRF object
obj = historical_CRF(client, proj)

# Upload CSV to Dataiku managed folder
obj.input_folder.put_file("/output.csv", io.BytesIO(csv_bytes))
print("✅ CSV uploaded successfully to managed folder as /output.csv")

# (Optional) Save locally too
with open("output.csv", "wb") as f:
    f.write(csv_bytes)
print("✅ CSV also saved locally as output.csv")


[{'path': '/3', 'size': 630, 'lastModified': 1758528240000}, {'path': '/323-201-00002Protocol.docx', 'size': 186925, 'lastModified': 1759820250000}, {'path': '/Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1) (4).pdf', 'size': 2544020, 'lastModified': 1759860489000}, {'path': '/Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1).pdf', 'size': 2544020, 'lastModified': 1757496080000}, {'path': '/ECS_output.xlsx', 'size': 2635330, 'lastModified': 1759156409000}, {'path': '/output.csv', 'size': 107472, 'lastModified': 1760369556000}, {'path': '/output.xlsx', 'size': 57081, 'lastModified': 1760337547000}, {'path': '/soa_1.pdf', 'size': 1178968, 'lastModified': 1759826677000}, {'path': '/soatest1.pdf', 'size': 1178968, 'lastModified': 1759763806000}, {'path': '/soatest2.pdf', 'size': 1178968, 'lastModified': 1759819989000}, {'path': '/standardECS.xlsx', 'size': 267281, 'lastModified': 1758524961000}, {'path': '/standard_ECS_output.xlsx', 'size': 267282, 'lastModified

In [76]:
# Count how many rows came from LLM
# print(final_df2['source'])
llm_count = (final_df2['source'] == "LLM Generated").sum()
print("Number of LLM generated rows:", llm_count)

# Count how many rows came from LLM
llm_count = (final_df2['source'] == "Fuzzy Match (Standard)").sum()
print("Number of Fuzzy Match rows:", llm_count)

llm_count = (final_df2['source'] == "Fuzzy Match (Historical)").sum()
print("Number of Historical Fuzzy Match rows:", llm_count)

llm_count = (final_df2['source'] == "Not Found").sum()
print("Number of Not Found:", llm_count)

print("length of sub_data",len(output_crfs["result"]))


Number of LLM generated rows: 805
Number of Fuzzy Match rows: 62
Number of Historical Fuzzy Match rows: 28
Number of Not Found: 0
length of sub_data 895


In [77]:
llm_count = final_df2[
    final_df2['source'].isin(["Fuzzy Match (Historical)", "Fuzzy Match (Standard)"])
]
display(llm_count)
print(f"Number of LLM rows: {len(llm_count)}")


,form_field_value,ecs_id,form_id,original_form_name,original_field,score,source,validation_id,form_name,validation_logic,reasoning,action,action_details
3,Total number of aborted attempts.,d6f6eb1b-ee4a-4975-af12-c2731fb43b3b,3e5a80e4-c245-440f-8496-185e690ba436,Date of Visit,Visit date,0.500000,Fuzzy Match (Historical),None,C-SSRS - Since Last Visit,Query if the total number of aborted attempts ...,None,,
4,Was informed consent obtained?,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent obtained?,0.642857,Fuzzy Match (Standard),MVAL_DS_IC006,Informed Consent,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>
5,Date of Consent,81661bed-0ab1-46b7-8f75-712d2402ca4b,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent date,0.555556,Fuzzy Match (Standard),MVAL_DS_IC001,Informed Consent,"(DS_IC.IFCOCCUR == ""Yes"") then (DS_IC.DSSTDAT_...",If informed consent was signed then Date of Co...,DS_IC.DSSTDAT_IC is enterable,<n/a>
6,Time of Consent,3d934727-4bd1-4033-97a2-d953f5361bf0,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent time,0.555556,Fuzzy Match (Standard),MVAL_DS_IC002,Informed Consent,"(DS_IC.IFCOCCUR == ""Yes"") then (DS_IC.DSSTTIM_...",If informed consent was signed then Time of Co...,DS_IC.DSSTTIM_IC is enterable,<n/a>
10,Protocol Version Number,426a6d30-e7db-407b-b10e-9c89ca760ce3,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Protocol version,0.769231,Fuzzy Match (Standard),MVAL_DS_IC005,Informed Consent,(DS_IC.QVAL_PROTVER is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>
...,...,...,...,...,...,...,...,...,...,...,...,...,...
731,Anatomical Location,1a2a14fe-a1b9-430b-8abd-57492df2b48b,151904be-561c-4ad7-8b84-0e308fb89a96,Prior/Concomitant Medications,AE indication 3,0.529412,Fuzzy Match (Standard),MVAL_CM015,Prior and Concomitant Medications,"(CM.CMYN_XX == ""Yes"") then (CM.CMLOC is entera...",If CM's were taken then this field is a valid ...,CM.CMROUTE is enterable,<n/a>
735,Specify Route,35d71559-c1e8-4336-abe8-c870ba617ef4,151904be-561c-4ad7-8b84-0e308fb89a96,Prior/Concomitant Medications,"If other, specify",0.666667,Fuzzy Match (Standard),MVAL_CM033,Prior and Concomitant Medications,"(CM.CMROUTE == ""OTHER "") then (CM.QVAL_CMROUTE...",If frequency is other then specify frequency i...,CM.QVAL_CMROUTEO is enterable,<n/a>
772,Was the procedure performed?,13fe5371-c28d-444a-b28f-6af33db4d721,c060516c-7bbd-4a27-b428-e98d9fdeb55f,Weight (Uns),Date,0.500000,Fuzzy Match (Historical),None,Weight / BMI,Value should = Y,None,Must Equal Y,"Procedure is expected to be performed, please ..."
773,Was the procedure performed?,13fe5371-c28d-444a-b28f-6af33db4d721,c060516c-7bbd-4a27-b428-e98d9fdeb55f,Weight (Uns),Time,0.500000,Fuzzy Match (Historical),None,Weight / BMI,Value should = Y,None,Must Equal Y,"Procedure is expected to be performed, please ..."


Number of LLM rows: 90


In [54]:
not_found_rows = final_df2[final_df2['source'] == "LLM Generated"]

found_unique = not_found_rows["original_form_name"].unique()

display(not_found_rows)


,form_field_value,ecs_id,form_id,original_form_name,original_field,score,source,validation_id,form_name,validation_logic,reasoning,action,action_details
0,Site ID,None,None,Enrollment,Site ID,NaN,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN
1,Participant ID,None,None,Enrollment,Participant ID,NaN,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN
2,Participant Number (Derived),None,None,Enrollment,Participant Number (Derived),NaN,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN
3,Visit date,None,None,Date of Visit,Visit date,NaN,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN
4,Informed consent obtained?,9ded7642-0302-47aa-a92d-0efe03e94e8e,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent obtained?,0.439024,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN
7,Derived date,9ded7642-0302-47aa-a92d-0efe03e94e8e,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Derived date,0.370370,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN
8,Informed consent version number,f8595a68-fb02-4c0e-b2e6-a96e05d1c1c0,47e7b2c2-e8e9-4a90-b974-954037f3b624,Informed Consent,Informed consent version number,0.478261,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN
9,Standardized disposition term,9ded7642-0302-47aa-a92d-0efe03e94e8e,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Standardized disposition term,0.272727,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN
10,Protocol version,9ded7642-0302-47aa-a92d-0efe03e94e8e,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Protocol version,0.322581,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN
15,"If Female, is the participant of childbearing ...",93f0a089-a8f5-4df8-8715-bfb2f1f6ca73,4a5552de-4909-4846-bd25-3c28f9351b53,Demographics,"If Female, is the participant of childbearing ...",0.281690,LLM Generated,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
# final_df.to_excel()
import pandas as pd
import io

# Sample DataFrame
df = final_df2

# Convert to Excel bytes
buffer = io.BytesIO()
with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Sheet1")

excel_bytes = buffer.getvalue()  # <-- This is your Excel file as bytes

obj.input_folder.put_file("/standardECS.xlsx",excel_bytes)

# Example: write to disk (optional)
with open("output.xlsx", "wb") as f:
    f.write(excel_bytes)
